In [2]:
# ==============================================================================
# CELL 0: ENVIRONMENT SETUP & SECURE API AUTHENTICATION
# Run this cell first. Installs the Google GenAI SDK, Pydantic, and dotenv.
# ==============================================================================
!pip install -q -U google-genai pydantic python-dotenv tabulate

import os
import sys
import time
import json
import random
import getpass
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field

# Official Google GenAI SDK
from google import genai
from google.genai import types
from google.genai.errors import APIError
from google.colab import userdata
userdata.get('GEMINI_API_KEY')
# ------------------------------------------------------------------------------
# Secure Gemini API Key Ingestion (Zero-Hardcoding Policy)
# ------------------------------------------------------------------------------
# Never hardcode API keys directly in scripts!
# In Google Colab, use the Secrets Manager (🔑 icon on the left panel) as 'GEMINI_API_KEY'.
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("🔑 Enter your Google Gemini API Key: ")
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

# Initialize Client
client = genai.Client(api_key=GEMINI_API_KEY)
print("✅ Google Gemini API Client initialized successfully!")

✅ Google Gemini API Client initialized successfully!


In [3]:
# ==============================================================================
# SECTION 1: API ARCHITECTURE, STATELESSNESS & SECURITY HYGIENE
# ==============================================================================
"""
1. WHY API KEYS MUST NEVER BE COMMITTED TO GITHUB:
   - Automated scrapers search public GitHub commits 24/7 for exposed API keys.
   - Leaked keys result in quota exhaustion, unexpected billing charges, and credential revocation.
   - DEFENSE: Store keys in a local `.env` file and add `.env` to `.gitignore`.

2. THE STATELESSNESS MENTAL MODEL:
   - LLMs are 100% STATELESS: The model remembers NOTHING between individual API calls.
   - To build a conversational multi-turn chatbot, YOU (the developer) must maintain a history list
     of previous (User, Model) turns and pass the cumulative array on every subsequent call.

3. MESSAGE ROLES MAPPING:
   ┌──────────────────────┬─────────────────────────┬───────────────────────────┐
   │ Role Type            │ OpenAI / Anthropic      │ Google Gemini API         │
   ├──────────────────────┼─────────────────────────┼───────────────────────────┤
   │ System Persona/Rules │ role: 'system'          │ config.system_instruction │
   │ User Message         │ role: 'user'            │ role: 'user'              │
   │ Model Response       │ role: 'assistant'       │ role: 'model'             │
   └──────────────────────┴─────────────────────────┴───────────────────────────┘
"""

"\n1. WHY API KEYS MUST NEVER BE COMMITTED TO GITHUB:\n   - Automated scrapers search public GitHub commits 24/7 for exposed API keys.\n   - Leaked keys result in quota exhaustion, unexpected billing charges, and credential revocation.\n   - DEFENSE: Store keys in a local `.env` file and add `.env` to `.gitignore`.\n\n2. THE STATELESSNESS MENTAL MODEL:\n   - LLMs are 100% STATELESS: The model remembers NOTHING between individual API calls.\n   - To build a conversational multi-turn chatbot, YOU (the developer) must maintain a history list\n     of previous (User, Model) turns and pass the cumulative array on every subsequent call.\n\n3. MESSAGE ROLES MAPPING:\n   ┌──────────────────────┬─────────────────────────┬───────────────────────────┐\n   │ Role Type            │ OpenAI / Anthropic      │ Google Gemini API         │\n   ├──────────────────────┼─────────────────────────┼───────────────────────────┤\n   │ System Persona/Rules │ role: 'system'          │ config.system_instruction │\

In [4]:
# ==============================================================================
# SECTION 2: PRE-FLIGHT TOKEN COUNTING & FINANCIAL COST ESTIMATION (UPDATED)
# ==============================================================================
"""
COST ESTIMATION BEST PRACTICE:
Count input tokens BEFORE invoking expensive generation calls to protect budget thresholds.
"""
import numpy as np
def preflight_cost_estimate(
    text_prompt: str,
    model_name: str = "gemini-3.6-flash",
    expected_output_tokens: int = 500
) -> Dict[str, Any]:
    """Calculates exact input tokens and estimates financial cost before calling the API."""
    # Count tokens using official Gemini Tokenizer
    token_resp = client.models.count_tokens(model=model_name, contents=text_prompt)
    input_tokens = token_resp.total_tokens

    # Official Rates per 1M tokens (USD)
    pricing = {
        "gemini-3.6-flash": {"in": 0.075, "out": 0.30},
        "gemini-1.5-pro":   {"in": 1.25,  "out": 5.00}
    }
    rate = pricing.get(model_name, pricing["gemini-3.6-flash"])

    est_cost = (input_tokens / 1e6 * rate["in"]) + (expected_output_tokens / 1e6 * rate["out"])

    return {
        "model": model_name,
        "input_tokens": input_tokens,
        "estimated_output_tokens": expected_output_tokens,
        "estimated_cost_usd": np.round(est_cost, 6),
        "cost_per_10k_calls": np.round(est_cost * 10000, 2)
    }

sample_payload = "Please summarize the last 10 quarterly financial filings of Apple, Microsoft, and Google."
# Updated to gemini-3.6-flash
estimate = preflight_cost_estimate(sample_payload, model_name="gemini-3.6-flash")
print("=== 📊 PRE-FLIGHT TOKEN & COST AUDIT ===")
for k, v in estimate.items():
    print(f"• {k:25s}: {v}")

=== 📊 PRE-FLIGHT TOKEN & COST AUDIT ===
• model                    : gemini-3.6-flash
• input_tokens             : 19
• estimated_output_tokens  : 500
• estimated_cost_usd       : 0.000151
• cost_per_10k_calls       : 1.51


In [5]:
# ==============================================================================
# SECTION 3: PRODUCTION RESILIENCE — EXPONENTIAL BACKOFF & RETRY LOOP
# ==============================================================================
"""
HANDLING API FAILURES IN PRODUCTION:
1. Rate Limits (HTTP 429 / ResourceExhausted): Hit requests-per-minute (RPM) ceiling.
2. Transient Server Errors (HTTP 500 / 503): Temporary Google Cloud infrastructure hiccup.
3. Network Timeouts: Connection dropped during streaming.

REMEDY: EXPONENTIAL BACKOFF WITH JITTER:
Wait time = (base_delay * 2^attempt) + random_jitter
Prevents "Thundering Herd" problem where all failed clients retry at the exact same millisecond.
"""

def execute_with_exponential_backoff(
    api_call_func,
    max_retries: int = 4,
    base_delay: float = 1.5
):
    """Wraps an API call in an exponential backoff retry loop with random jitter."""
    for attempt in range(max_retries):
        try:
            return api_call_func()
        except APIError as e:
            if attempt == max_retries - 1:
                print(f"❌ Max retries reached. Fatal API Error: {e}")
                raise e
            # Calculate backoff delay with jitter
            delay = (base_delay * (2 ** attempt)) + random.uniform(0.1, 0.8)
            print(f"⚠️ Warning: Transient API Error ({e.code}). Retrying in {delay:.2f}s... (Attempt {attempt+1}/{max_retries})")
            time.sleep(delay)

In [7]:
# ==============================================================================
# SECTION 4: REUSABLE GEMINI WRAPPER & 3-TURN CHAT
# ==============================================================================
# Construct a reusable production function supporting:
# - Streaming responses (Low Time-To-First-Token)
# - System instructions
# - Dynamic temperature
# - Exponential backoff retry logic

def gemini_call(
    prompt: str,
    system_instruction: str = "You are a concise, helpful enterprise AI assistant.",
    temperature: float = 0.2,
    stream: bool = False,
    model: str = "gemini-3.6-flash"
) -> str:
    """Production-grade wrapper for Google Gemini API with error handling and streaming."""
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=temperature,
        max_output_tokens=800
    )

    if stream:
        def stream_call():
            full_text = []
            response_stream = client.models.generate_content_stream(
                model=model, contents=prompt, config=config
            )
            for chunk in response_stream:
                if chunk.text:
                    print(chunk.text, end="", flush=True)
                    full_text.append(chunk.text)
            print() # Print final newline
            return "".join(full_text)

        return execute_with_exponential_backoff(stream_call)
    else:
        def standard_call():
            resp = client.models.generate_content(
                model=model, contents=prompt, config=config
            )
            return resp.text.strip()

        return execute_with_exponential_backoff(standard_call)

# ------------------------------------------------------------------------------
# 3-Turn Conversational Memory Loop Demonstration
# ------------------------------------------------------------------------------
print("=== MULTI-TURN CONVERSATION LOOP ===")

# Explicitly maintain stateless conversation history
conversation_history = []
system_persona = "You are a Senior PostgreSQL Database Administrator. Answer concisely in 2 sentences."

def send_chat_turn(user_message: str):
    print(f"\n👤 User: {user_message}")
    print("🤖 Assistant: ", end="")

    # 1. Append user message to history
    conversation_history.append({"role": "user", "parts": [{"text": user_message}]})

    # 2. Call Gemini passing full conversation history
    config = types.GenerateContentConfig(
        system_instruction=system_persona,
        temperature=0.0
    )
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=conversation_history,
        config=config
    )

    bot_reply = response.text.strip()
    print(bot_reply)

    # 3. Append model response to history to maintain context
    conversation_history.append({"role": "model", "parts": [{"text": bot_reply}]})

# Execute 3-Turn Dialogue (Demonstrating Context Memory)
send_chat_turn("What is the difference between a clustered and non-clustered index?")
send_chat_turn("Which one is faster for range queries on primary keys?") # Pronoun resolution!
send_chat_turn("Can a table have multiple of the faster one?")           # Contextual follow-up!

=== MULTI-TURN CONVERSATION LOOP ===

👤 User: What is the difference between a clustered and non-clustered index?
🤖 Assistant: A clustered index physically dictates the storage order of the table's data rows on disk, restricting a table to only one such index. In contrast, a non-clustered index maintains a separate logical structure containing indexed keys and pointers to the actual data rows, allowing a table to have multiple instances.

👤 User: Which one is faster for range queries on primary keys?
🤖 Assistant: ⚠️ Warning: Transient API Error (503). Retrying in 2.33s... (Attempt 1/4)
⚠️ Warning: Transient API Error (503). Retrying in 4.16s... (Attempt 2/4)
A clustered index is significantly faster for range queries because contiguous key values are physically stored together on the same disk pages. This alignment allows the database to perform efficient sequential I/O reads, whereas a non-clustered index requires costly, random page lookups for every row in the range.

👤 User: Can a 

'No, a table can only have one clustered index because data rows can physically exist in only one sorted order on disk. In PostgreSQL, while you can build multiple indexes, the physical table can only be ordered (`CLUSTER`ed) based on a single index at any given time.'

In [9]:
# ==============================================================================
# SECTION 5: STUDENT LAB WORKSPACE (PORTFOLIO APPLICATION)
# ==============================================================================
"""
🎓 STUDENT LAB ASSIGNMENT:
Build an end-to-end AI Application: "The Executive Resume Bullet & Impact Optimizer"

APPLICATION REQUIREMENTS:
1. Structured JSON Schema (Pydantic):
   - `original_bullet`: Raw user text
   - `xyz_formatted_bullet`: Rewritten using Google's XYZ Formula:
     "Accomplished [X], as measured by [Y], by doing [Z]"
   - `impact_metric`: The quantifiable numeric KPI
   - `action_verb`: Strong opening action verb
   - `seniority_score`: Integer rating (1 to 10) of executive presence
   - `critique`: 1-sentence explanation of what was improved
2. Interactive Revision History: Allow user to request a revision (multi-turn).
3. Streaming or Schema Parsing: Correctly parse and display output.
4. Error Handling: Enclose calls in retry blocks.
"""

# ==============================================================================
# TASK 1: DEFINE PYDANTIC SCHEMA FOR STRUCTURED RESUME OPTIMIZATION
# ==============================================================================

# TODO 1.1: Complete the Pydantic Schema
class ResumeOptimization(BaseModel):
    """Structured output returned by the Executive Resume Bullet Optimizer."""

    original_bullet: str = Field(
        description="The user's original resume bullet exactly as provided."
    )
    xyz_formatted_bullet: str = Field(
        description='Rewritten using the XYZ Formula: "Accomplished [X], as measured by [Y], by doing [Z]."'
    )
    impact_metric: str = Field(
        description="The most important quantifiable KPI. If no numeric KPI is provided, return 'Not provided' and never invent one."
    )
    action_verb: str = Field(
        description="A strong action verb used to open the optimized resume bullet."
    )
    seniority_score: int = Field(
        ge=1,
        le=10,
        description="Integer rating from 1 to 10 representing executive presence."
    )
    critique: str = Field(
        description="Exactly one sentence explaining what was improved."
    )


# ==============================================================================
# TASK 2: BUILD THE APPLICATION ENGINE
# ==============================================================================

def optimize_resume_bullet(
    bullet: str,
    revision_request: Optional[str] = None,
    previous_result: Optional[ResumeOptimization] = None,
    model: str = "gemini-3.6-flash"
) -> ResumeOptimization:
    """Optimize one resume bullet and return a validated Pydantic object."""

    if not isinstance(bullet, str) or not bullet.strip():
        raise ValueError("Resume bullet must be a non-empty string.")

    revision_context = ""
    if previous_result is not None:
        revision_context = f"""
Previous optimized result:
{previous_result.model_dump_json(indent=2)}

Revision request:
{revision_request or "Improve the previous version while preserving factual accuracy."}
"""

    prompt = f"""
You are an executive resume editor.

Transform the following resume bullet using the XYZ Formula:
"Accomplished [X], as measured by [Y], by doing [Z]".

Rules:
1. Preserve the factual meaning of the original bullet.
2. Never invent a percentage, dollar amount, headcount, time saving, revenue figure,
   or other KPI that is not supported by the original text or revision request.
3. If no numeric KPI is available, use "Not provided" for impact_metric.
4. Use a strong action verb at the beginning of xyz_formatted_bullet.
5. seniority_score must be an integer from 1 through 10.
6. critique must contain exactly one sentence.
7. Return only structured JSON matching the supplied response schema.

Original bullet:
{bullet.strip()}
{revision_context}
"""

    # TODO 2.1: Configure GenerateContentConfig with temperature=0.1, response_mime_type='application/json', and response_schema
    config = types.GenerateContentConfig(
        system_instruction=(
            "You are a precise executive resume optimization assistant. "
            "Return only valid structured JSON that follows the supplied schema."
        ),
        temperature=0.1,
        response_mime_type="application/json",
        response_schema=ResumeOptimization,
        max_output_tokens=800,
    )

    # TODO 2.2: Execute API call with exponential backoff
    def api_call():
        response = client.models.generate_content(
            model=model,
            contents=prompt,
            config=config,
        )

        if not response.text:
            raise ValueError("Gemini returned an empty response.")

        # Prefer SDK parsed output when available; otherwise validate the JSON text.
        if getattr(response, "parsed", None) is not None:
            parsed = response.parsed
            if isinstance(parsed, ResumeOptimization):
                return parsed
            return ResumeOptimization.model_validate(parsed)

        return ResumeOptimization.model_validate_json(response.text)

    return execute_with_exponential_backoff(api_call)


def revise_resume_bullet(
    current_result: ResumeOptimization,
    revision_request: str,
    model: str = "gemini-3.6-flash"
) -> ResumeOptimization:
    """Convenience function for multi-turn revision of an existing result."""
    if not revision_request.strip():
        raise ValueError("Revision request must be a non-empty string.")

    return optimize_resume_bullet(
        bullet=current_result.original_bullet,
        revision_request=revision_request,
        previous_result=current_result,
        model=model,
    )


def display_resume_result(result: ResumeOptimization) -> None:
    """Print a structured optimization result in a readable format."""
    print("\n=== EXECUTIVE RESUME OPTIMIZATION ===")
    print(f"Original bullet      : {result.original_bullet}")
    print(f"XYZ formatted bullet : {result.xyz_formatted_bullet}")
    print(f"Impact metric        : {result.impact_metric}")
    print(f"Action verb          : {result.action_verb}")
    print(f"Seniority score      : {result.seniority_score}/10")
    print(f"Critique             : {result.critique}")


# ==============================================================================
# TASK 3: TEST APPLICATION ON REAL-WORLD WEAK BULLETS
# ==================================================================
# Three realistic weak bullets covering missing metrics, qualitative impact,
# and an already-quantified achievement.
test_bullets = [
    "Managed a team and improved the reporting process.",
    "Worked on a customer support dashboard that helped the team resolve issues faster.",
    "Reduced monthly reporting time by 30% by automating Excel-based data preparation."
]

# Uncomment the following block to run live Gemini tests after Cell 0 has
# successfully initialized the client and your API quota is available.
#
# for i, weak_bullet in enumerate(test_bullets, start=1):
#     print(f"\n\n### TEST BULLET {i}")
#     result = optimize_resume_bullet(weak_bullet)
#     display_resume_result(result)
#
#     # Demonstrate multi-turn revision history on the first test result.
#     if i == 1:
#         revised = revise_resume_bullet(
#             result,
#             "Make the bullet sound more strategic and suitable for a senior manager, "
#             "but do not invent any metrics."
#         )
#         print("\n### REVISED VERSION")
#         display_resume_result(revised)

# ==============================================================================
# SECTION 6: GIT REPOSITORY HYGIENE — CREATING .ENV AND .GITIGNORE
# ==============================================================================
"""
INSTRUCTIONS FOR PUSHING TO GITHUB SAFELY:

1. Create a `.env` file locally:
   GEMINI_API_KEY=your_actual_key_here

2. Create a `.gitignore` file in your project root containing:
   .env
   .env.local
   *.joblib
   __pycache__/
   .ipynb_checkpoints/

3. In your Python script (`app.py`), load the key cleanly via:
   from dotenv import load_dotenv
   load_dotenv()
   api_key = os.getenv("GEMINI_API_KEY")
"""

# Script to generate .gitignore locally in Colab
with open(".gitignore", "w") as f:
    f.write(".env\n.env.*\n*.joblib\n__pycache__/\n.ipynb_checkpoints/\n")

print("✅ '.gitignore' template created successfully!")